In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing 
from sklearn.preprocessing import LabelEncoder
%matplotlib inline

from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, precision_score

from sklearn import model_selection
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB,BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

from sklearn import datasets, linear_model, metrics
from sklearn.model_selection import GridSearchCV
import sklearn.model_selection as ms
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

### load data

In [2]:
data = pd.read_csv("./RNA_seq.csv")
dataT=np.array(data)
data=dataT.T
co=data[0]
data1=np.delete(data,0,axis=0)
data=data1
datadf= pd.DataFrame(data=data[0:,0:],columns=co)
datadf.head()
data=datadf
data= data.replace("NOTLC",value=0)
data= data.replace("LC",value=1)
X_yuan=data.drop(['Group'],axis=1)
y_yuan=data['Group']

### LOF

In [3]:
### LOF
df = data
df = df.dropna()
df = df.drop_duplicates()
from sklearn.neighbors import LocalOutlierFactor
lof = LocalOutlierFactor(n_neighbors=10, contamination=0.1)  # contamination 为异常值比例
lof_predictions = lof.fit_predict(df)
# 将预测结果添加到 DataFrame 中
df['LOF_Predictions'] = lof_predictions
# 筛选出离群值（预测结果为 -1 的点）
outliers = df[df['LOF_Predictions'] == -1]
print("离群值：")
print(outliers)
# 删除离群值所在的行
df_cleaned = df[df['LOF_Predictions'] == 1]  # 保留正常值
df_cleaned = df_cleaned.drop(columns=['LOF_Predictions'])  # 删除辅助列
print("删除离群值后的数据：")
print(df_cleaned)
df = df_cleaned
X=df.drop(['Group'],axis=1)
y=df['Group']

离群值：
      Group ENSG00000000003.15 ENSG00000000005.6 ENSG00000000419.13  \
8         1               7472                 1               4818   
26        1                791                 0                881   
31        1               1339                 0               3032   
37        1               1097                 1               1486   
56        1               2020                 1               1419   
...     ...                ...               ...                ...   
1027      1              10623                 3               4224   
1044      1               9468                 1               2420   
1055      1               4513                 0               4348   
1076      1               8401                 2               4145   
1081      1               2065                 0               5496   

     ENSG00000000457.14 ENSG00000000460.17 ENSG00000000938.13  \
8                   720                931                385   
26          

### standard

In [4]:
### standard
column=X.columns
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_pre=sc.fit_transform(X)
X_pre=pd.DataFrame(data=X_pre,columns=column)

In [5]:
X_pre=pd.DataFrame(data=X_pre,columns=column)

### train-test

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X_pre,y,test_size=0.2,random_state=42,stratify=None)
print(f'Train shape : {X_train.shape}\nTest shape: {X_test.shape}')

Train shape : (790, 60660)
Test shape: (198, 60660)


### grid search

In [6]:
#网格搜索
def gridsearch(params,estimator,Xtrain,ytrain,cvnumber):
    #print("params:",params)
    model = ms.GridSearchCV(estimator, params, cv=cvnumber)
    model.fit(Xtrain, ytrain)
    print("the best_params of the model is:",model.best_params_)
    print("the best_score of the model is:",model.best_score_)
    print("the best_estimator of the model is:",model.best_estimator_)

In [ ]:
#RF
params={
    'n_estimators':np.arange(1,21,1),
    'max_depth': np.arange(1,21,1),
}
model = ms.GridSearchCV(RandomForestClassifier(), params, cv=10)
model.fit(X_train, y_train)

print("the best_params of the RF model is:",model.best_params_)
# 获取最优得分
print("the best_score of the RF model is:",model.best_score_)
# 获取最优模型的信息
print("the best_estimator of the RF model is:",model.best_estimator_)

In [ ]:
#AdaBoost
params={
    'n_estimators':[10,50,100,200,300,400,500],
    'learning_rate':[0.0001,0.001,0.01,0.1,1]
}
model = ms.GridSearchCV(AdaBoostClassifier(), params, cv=10)
model.fit(X_train, y_train)

print("the best_params of the AdaBoost model is:",model.best_params_)
# 获取最优得分
print("the best_score of the AdaBoost model is:",model.best_score_)
# 获取最优模型的信息
print("the best_estimator of the AdaBoost model is:",model.best_estimator_)

In [8]:
import sklearn.model_selection as ms
#GausianNB
nb = GaussianNB()
param_grid = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}
# 使用GridSearchCV进行网格搜索
model=ms.GridSearchCV(nb, param_grid=param_grid, cv=10)
model.fit(X_train, y_train)
print("the best_params of the XGBoost model is:",model.best_params_)
# 获取最优得分
print("the best_score of the XGBoost model is:",model.best_score_)
# 获取最优模型的信息
print("the best_estimator of the XGBoost model is:",model.best_estimator_)

the best_params of the XGBoost model is: {'var_smoothing': 1e-09}
the best_score of the XGBoost model is: 0.9009273772204806
the best_estimator of the XGBoost model is: GaussianNB()


In [1]:
#XGBoost
params={
    'n_estimators':[10,50,100,200,300,400,500],
    'learning_rate':[0.001,0.01,0.05,0.1,0.3,1],
    'max_depth': [5, 10, 15, 20, 25]
}
model = ms.GridSearchCV(XGBClassifier(), params, cv=10)
model.fit(X_train, y_train)
print("the best_params of the XGBoost model is:",model.best_params_)
# 获取最优得分
print("the best_score of the XGBoost model is:",model.best_score_)
# 获取最优模型的信息
print("the best_estimator of the XGBoost model is:",model.best_estimator_)

In [ ]:
#decisiontree
from sklearn.tree import DecisionTreeClassifier
params={
    'max_depth':np.arange(1,13,1),
    'min_samples_split': np.arange(2,7,2),
    'min_samples_leaf':np.arange(1,5,1)
}
gridsearch(params,DecisionTreeClassifier(),X_train,y_train,10)